<h1>Validación 06. Consistencia vertical entre el DEM y el perfil geológico 3D</h1>

<h2>Objetivo</h2>

<p>
El objetivo de este notebook es comprobar directamente si las elevaciones almacenadas
en los vértices de la geometría geológica 3D generada por SecGeol coinciden con las
elevaciones del modelo digital de elevación en sus respectivas coordenadas XY.
</p>

<p>
Esta validación permitirá determinar si el perfil 3D se encuentra realmente apoyado
sobre la superficie del DEM o si existe alguna diferencia vertical entre ambos
productos.
</p>

<h2>Datos de entrada</h2>

<ul>
    <li>
        <strong>Modelo digital de elevación:</strong>
        <code>C:\Proyectos\2026\seccion\dem2.tif</code>
    </li>
    <li>
        <strong>Polígonos geológicos 3D generados por SecGeol:</strong>
        <code>C:\Proyectos\2026\seccion\salidas\poli3.shp</code>
    </li>
</ul>

<h2>Procedimiento</h2>

<p>
Para cada vértice de la geometría 3D se realizará la siguiente comparación:
</p>

<ol>
    <li>
        Se obtendrán sus coordenadas X, Y y Z.
    </li>
    <li>
        Se consultará el valor del DEM en la posición XY del vértice mediante
        <code>provider.sample()</code>.
    </li>
    <li>
        Se comparará la elevación almacenada en la geometría 3D con la elevación
        obtenida directamente del DEM.
    </li>
</ol>

<p style="text-align:center;">
    <strong>&Delta;Z = Z<sub>perfil 3D</sub> - Z<sub>DEM</sub></strong>
</p>

<h2>Indicadores de validación</h2>

<ul>
    <li>Número total de vértices evaluados.</li>
    <li>Número de muestras válidas e inválidas.</li>
    <li>Diferencia vertical mínima.</li>
    <li>Diferencia vertical máxima.</li>
    <li>Diferencia media absoluta.</li>
    <li>RMSE vertical.</li>
</ul>

<div style="
border-left:6px solid #2E75B6;
background:#F4F8FC;
padding:14px;
margin-top:18px;
">

<h3 style="margin-top:0;">Resultado esperado</h3>

<div>
Si las elevaciones del perfil 3D coinciden con las obtenidas directamente del DEM,
las diferencias verticales deberán ser nulas o encontrarse únicamente dentro de la
precisión numérica del sistema.
</div>

<div style="margin-top:12px;">
Un resultado de este tipo confirmará que la geometría tridimensional generada por
SecGeol se encuentra verticalmente consistente con el modelo digital de elevación
y que cualquier aparente separación observada en la vista 3D no se origina en los
valores Z almacenados en la geometría.
</div>

</div>

### <span style="color:#cc416d">1.Importaciones</span>

In [9]:
import math
import numpy as np
from collections import defaultdict
from qgis.core import (
    QgsApplication,
    QgsVectorLayer,
    QgsRasterLayer,
    QgsPointXY, QgsGeometry
)

### <span style="color:#cc416d">2. Carga de datos</span>

In [4]:
ruta_dem = r"C:\Proyectos\2026\seccion\dem2.tif"

ruta_poligonos_3d = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\poli3.shp"
)

dem_layer = QgsRasterLayer(
    ruta_dem,
    "DEM"
)
poligonos_3d_layer = QgsVectorLayer(
    ruta_poligonos_3d,
    "Poligonos_3D_SecGeol",
    "ogr"
)


### <span style="color:#cc416d">3. Celda siguiente: extraer todos los vértices</span>

In [5]:
provider_dem = dem_layer.dataProvider()

vertices_por_xy = defaultdict(list)

total_vertices = 0

for feat in poligonos_3d_layer.getFeatures():
    for pt in feat.geometry().vertices():
        total_vertices += 1

        # Redondeo únicamente para agrupar vértices que comparten
        # la misma posición XY dentro de la precisión del shapefile
        clave_xy = (
            round(pt.x(), 8),
            round(pt.y(), 8)
        )

        vertices_por_xy[clave_xy].append(
            {
                "x": pt.x(),
                "y": pt.y(),
                "z": pt.z()
            }
        )

print("Vértices totales:", total_vertices)
print("Posiciones XY únicas:", len(vertices_por_xy))

Vértices totales: 1182
Posiciones XY únicas: 1166


### <span style="color:#cc416d">4. Seleccionar la elevación superior en cada XY</span>

In [6]:
vertices_superficie = []

for clave_xy, grupo in vertices_por_xy.items():

    vertice_superior = max(
        grupo,
        key=lambda registro: registro["z"]
    )

    vertices_superficie.append(
        vertice_superior
    )

print(
    "Vértices candidatos de superficie:",
    len(vertices_superficie)
)

Vértices candidatos de superficie: 1166


### <span style="color:#cc416d">5. Comparar con el DEM</span> 

In [7]:
resultados_verticales = []
muestras_invalidas = []

for indice, vertice in enumerate(vertices_superficie):

    z_dem, ok = provider_dem.sample(
        QgsPointXY(
            vertice["x"],
            vertice["y"]
        ),
        1
    )

    if not ok or z_dem is None:
        muestras_invalidas.append({
            "indice": indice,
            **vertice
        })
        continue

    dz = vertice["z"] - z_dem

    resultados_verticales.append({
        "indice": indice,
        "x": vertice["x"],
        "y": vertice["y"],
        "z_perfil_3d": vertice["z"],
        "z_dem": z_dem,
        "dz": dz
    })

print("Muestras válidas:", len(resultados_verticales))
print("Muestras inválidas:", len(muestras_invalidas))

Muestras válidas: 1166
Muestras inválidas: 0


### <span style="color:#cc416d">6. Resumen inicial </span> 

In [10]:
dz_np = np.array(
    [r["dz"] for r in resultados_verticales],
    dtype=float
)

print("ΔZ mínimo:", dz_np.min())
print("ΔZ máximo:", dz_np.max())
print(
    "Diferencia media absoluta:",
    np.mean(np.abs(dz_np))
)
print(
    "RMSE:",
    np.sqrt(np.mean(dz_np ** 2))
)

ΔZ mínimo: -341.7900085449219
ΔZ máximo: 0.2916456830998868
Diferencia media absoluta: 0.6376099530844227
RMSE: 13.539628337287327


### <span style="color:#cc416d">7. Cuántos puntos tienen diferencia prácticamente nula </span> 

In [11]:
tolerancia = 1e-6

coincidentes = np.sum(
    np.abs(dz_np) <= tolerancia
)

print(
    "Coincidencias dentro de tolerancia:",
    coincidentes
)

print(
    "Porcentaje de coincidencia:",
    100.0 * coincidentes / len(dz_np)
)

Coincidencias dentro de tolerancia: 1155
Porcentaje de coincidencia: 99.05660377358491


### <span style="color:#cc416d">7. Vamos a encontrar al culpable </span> 

In [13]:
casos = [
    r for r in resultados_verticales
    if abs(r["dz"]) > 1
]

print("Casos encontrados:", len(casos))

for c in casos:
    print(c)

Casos encontrados: 5
{'indice': 0, 'x': 617237.798590572, 'y': 2113744.082300052, 'z_perfil_3d': 86.22000122070312, 'z_dem': 188.50999450683594, 'dz': -102.28999328613281}
{'indice': 82, 'x': 618337.7384063124, 'y': 2113469.7456026208, 'z_perfil_3d': 86.22000122070312, 'z_dem': 428.010009765625, 'dz': -341.7900085449219}
{'indice': 83, 'x': 618249.284225068, 'y': 2113491.8070168993, 'z_perfil_3d': 426.44813418610994, 'z_dem': 429.0299987792969, 'dz': -2.5818645931869355}
{'indice': 232, 'x': 620993.0286415742, 'y': 2112807.4879559395, 'z_perfil_3d': 86.22000122070312, 'z_dem': 380.260009765625, 'dz': -294.0400085449219}
{'indice': 1113, 'x': 622496.3634895857, 'y': 2112432.54024386, 'z_perfil_3d': 623.9711946977332, 'z_dem': 625.6500244140625, 'dz': -1.6788297163293464}


### <span style="color:#cc416d">8. Comprobaciones </span> 

In [14]:
for c in casos:

    print("\n----------------")

    print(f"x = {c['x']:.15f}")
    print(f"y = {c['y']:.15f}")
    print(f"z = {c['z_perfil_3d']:.15f}")


----------------
x = 617237.798590571968816
y = 2113744.082300052046776
z = 86.220001220703125

----------------
x = 618337.738406312419102
y = 2113469.745602620765567
z = 86.220001220703125

----------------
x = 618249.284225068055093
y = 2113491.807016899343580
z = 426.448134186109939

----------------
x = 620993.028641574201174
y = 2112807.487955939490348
z = 86.220001220703125

----------------
x = 622496.363489585695788
y = 2112432.540243859868497
z = 623.971194697733154


<h2>Conclusión</h2>

<p>
La comparación directa entre las elevaciones almacenadas en la geometría geológica
3D y los valores consultados en el modelo digital de elevación mostró una
correspondencia prácticamente total entre ambos productos.
</p>

<p>
De las posiciones evaluadas, <strong>1,155 coincidieron exactamente</strong> dentro
de la tolerancia numérica establecida, lo que representa aproximadamente
<strong>99.06 %</strong> de coincidencia.
</p>

<p>
Las diferencias detectadas se concentraron únicamente en cinco posiciones y no
corresponden a un desplazamiento vertical sistemático del perfil:
</p>

<ul>
    <li>
        Tres casos presentan una elevación de <strong>86.22 m</strong>, valor utilizado
        como cota inferior para cerrar los polígonos geológicos. Estos vértices forman
        parte de la base de la sección y no representan la superficie topográfica,
        por lo que no deben coincidir con la elevación del DEM.
    </li>
    <li>
        Los dos casos restantes presentan diferencias verticales aproximadas de
        <strong>2.58 m</strong> y <strong>1.68 m</strong>. Estas posiciones corresponden
        a vértices asociados con contactos o intersecciones geológicas, cuya elevación
        se obtiene mediante interpolación sobre la geometría del perfil y puede diferir
        ligeramente del valor discreto de la celda raster consultada en la misma posición.
    </li>
</ul>

<p>
Los valores extremos obtenidos en el cálculo global de la diferencia vertical y del
RMSE están dominados por los vértices inferiores de cierre. Por esta razón, dichos
indicadores no deben interpretarse como errores de la superficie topográfica del
perfil 3D.
</p>

<div style="
border-left:6px solid #2E75B6;
background:#F4F8FC;
padding:14px;
margin-top:18px;
">

<h3 style="margin-top:0;">Resultado de la validación</h3>

<div>
La envolvente superior del perfil geológico 3D coincide con la superficie del DEM
en prácticamente la totalidad de los puntos evaluados.
</div>

<div style="margin-top:12px;">
No se identificó una diferencia vertical sistemática entre las elevaciones del perfil
3D generado por SecGeol y los valores del modelo digital de elevación.
</div>

<div style="margin-top:12px;">
Las discrepancias restantes corresponden a elementos geométricos esperados en la
construcción de los polígonos, como los vértices inferiores de cierre y algunos puntos
interpolados en contactos geológicos.
</div>

<div style="margin-top:12px;">
En consecuencia, la geometría tridimensional producida por SecGeol puede considerarse
verticalmente consistente con el DEM. El aparente desfase observado durante la
visualización no se origina en los valores Z almacenados en la geometría ni en el
procedimiento de reconstrucción tridimensional.
</div>

</div>